# L55 · 信号向前 — 前向传播拆解

**目标**：为 `Value` 类补全减法、除法、乘幂（`__pow__`）、`tanh`、`relu`、`exp` 算子，使其能表达完整的前向计算图。

🔗 **Aurora 连接**：这些算子对应神经网络里的每一层——线性变换（`+`、`*`、`**`）加激活函数（`tanh`/`relu`），自动微分就建立在这套算子之上。见 `src/aurora/` 中未来的 `nn/` 子包。

每调用一个算子，就在计算图里新增一个节点，同时把"怎么把梯度传回来"封装进 `_backward` 闭包——前向和反向两件事同时完成。加、减、乘、除、幂次构成线性组合；`tanh`/`relu`/`exp` 是激活函数，它们的梯度公式直接用前向输出值 `out.data` 表达，避免重复计算。这节课实现这些缺失的算子，让 `Value` 足以描述一个完整的感知机前向传播。

In [ ]:
import math

# 完整的 Value 类（含加法、乘法基础，本节要补全其余算子）
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- 已有算子 ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self):   return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return Value(other) * self**-1

    # --- 待实现算子（占位，下面编码任务格会替换）---
    def __pow__(self, n): raise NotImplementedError
    def relu(self):       raise NotImplementedError
    def tanh(self):       raise NotImplementedError
    def exp(self):        raise NotImplementedError

    # --- 反向传播（拓扑序） ---
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

print("Value 类已定义")

## 概念 1：每个算子 = 新节点 + _backward 闭包

调用 `a * b` 时发生两件事：

1. **前向**：`out = Value(a.data * b.data)`，挂载 `_prev = {a, b}`
2. **后向闭包**：在 `_backward` 里写入链式法则——`a.grad += b.data * out.grad`

闭包捕获了 `a`、`b`、`out` 三个活对象，调用 `.backward()` 时按拓扑逆序执行，梯度自动累积到叶节点的 `.grad`。

In [ ]:
# 演示：加法节点的 _backward
x = Value(3.0)
y = Value(4.0)
z = x + y       # z.data = 7.0
z.grad = 1.0
z._backward()   # 手动触发
print(f"x.grad = {x.grad}, y.grad = {y.grad}")  # 应该都是 1.0

## 概念 2：tanh 梯度用前向输出值直接表达

`tanh(x) = (e^{2x} - 1) / (e^{2x} + 1)`，其导数：

```
d(tanh(x))/dx = 1 - tanh(x)^2
```

实现技巧：`out.data` 就是已经算好的 `tanh(x)` 值，反向直接写：

```python
self.grad += (1 - out.data**2) * out.grad
```

这样不用再调一次 `math.tanh`，也不用存中间变量。

In [ ]:
# 验证公式：数值差分 vs 解析梯度
import math

x_val = 0.8
h = 1e-5
analytic = 1 - math.tanh(x_val)**2
numeric  = (math.tanh(x_val + h) - math.tanh(x_val - h)) / (2*h)
print(f"解析梯度: {analytic:.6f}")
print(f"数值梯度: {numeric:.6f}")
print(f"误差: {abs(analytic - numeric):.2e}")

## 概念 3：ReLU 梯度（次梯度）

`relu(x) = max(0, x)`，在 `x > 0` 时梯度为 1，`x < 0` 时梯度为 0，`x = 0` 处次梯度取 0：

```
d(relu(x))/dx = 1  if x > 0 else 0
```

反向实现：

```python
self.grad += (out.data > 0) * out.grad
```

`out.data > 0` 是布尔值，Python 里等于 `1` 或 `0`，直接相乘即可。ReLU 比 tanh 快，但 `x < 0` 区间梯度死亡（Dead ReLU 问题，ELU/Leaky ReLU 是常见修复方案）。

In [ ]:
# 演示 ReLU 次梯度
vals = [-2.0, 0.0, 3.0]
for v in vals:
    grad = float(v > 0)
    print(f"relu'({v:+.1f}) = {grad}")

## 1. ✏️ 实现 `Value.__pow__`, `Value.relu`, `Value.tanh`

**推理路线**：
1. `__pow__(self, n)`：`out = Value(self.data**n)`；`_backward`：`self.grad += n * self.data**(n-1) * out.grad`（n 是普通数字，不是 Value）
2. `relu(self)`：`t = max(0, self.data)`；`out = Value(t)`；`_backward`：`self.grad += (t > 0) * out.grad`
3. `tanh(self)`：`t = math.tanh(self.data)`；`out = Value(t)`；`_backward`：`self.grad += (1 - t**2) * out.grad`
4. `exp(self)`（附加）：`t = math.exp(self.data)`；`_backward`：`self.grad += t * out.grad`（exp 的导数是自身）

**参考输入输出**：
```
Value(-2.0).__pow__(2).data  →  4.0
Value(-2.0).relu().data      →  0.0
Value( 0.5).relu().data      →  0.5
Value( 1.0).tanh().data      ≈  0.7616
Value(-2.0).tanh().data      ≈ -0.9640
Value( 1.0).exp().data       ≈  2.7183
```

<details><summary>点击查看参考实现</summary>

```python
def __pow__(self, n):
    assert isinstance(n, (int, float)), "__pow__ 只支持数字指数"
    out = Value(self.data**n, (self,), f'**{n}')
    def _backward():
        self.grad += n * (self.data ** (n - 1)) * out.grad
    out._backward = _backward
    return out

def relu(self):
    t = max(0, self.data)
    out = Value(t, (self,), 'relu')
    def _backward():
        self.grad += (t > 0) * out.grad
    out._backward = _backward
    return out

def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

def exp(self):
    t = math.exp(self.data)
    out = Value(t, (self,), 'exp')
    def _backward():
        self.grad += t * out.grad
    out._backward = _backward
    return out
```

</details>

In [ ]:
# ✏️ TODO：在下面完整实现四个算子，替换 Value 类里的占位 raise NotImplementedError

def __pow__(self, n):
    # ✏️ TODO
    assert isinstance(n, (int, float))
    out = Value(...)        # 计算 self.data**n
    def _backward():
        self.grad += ...    # n * self.data**(n-1) * out.grad
    out._backward = _backward
    return out

def relu(self):
    t = ...                 # ✏️ TODO: max(0, self.data)
    out = Value(t, (self,), 'relu')
    def _backward():
        self.grad += ...    # ✏️ TODO: (t > 0) * out.grad
    out._backward = _backward
    return out

def tanh(self):
    t = ...                 # ✏️ TODO: math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += ...    # ✏️ TODO: (1 - t**2) * out.grad
    out._backward = _backward
    return out

def exp(self):
    t = ...                 # ✏️ TODO: math.exp(self.data)
    out = Value(t, (self,), 'exp')
    def _backward():
        self.grad += ...    # ✏️ TODO: t * out.grad
    out._backward = _backward
    return out

# 绑定到 Value 类
Value.__pow__ = __pow__
Value.relu    = relu
Value.tanh    = tanh
Value.exp     = exp

In [ ]:
# 检查：基本前向值
assert Value(-2.0).__pow__(2).data == 4.0,   "__pow__ 前向错误"
assert Value(-2.0).relu().data == 0.0,        "relu 前向：负数应输出 0"
assert Value( 0.5).relu().data == 0.5,        "relu 前向：正数应透传"
assert abs(Value(1.0).tanh().data - 0.7616) < 1e-4, "tanh(1.0) 应 ≈ 0.7616"
assert abs(Value(1.0).exp().data  - math.e)  < 1e-5, "exp(1.0) 应 ≈ e"

# 检查：__pow__ 反向
a = Value(3.0)
b = a**2          # b = 9, db/da = 2*3 = 6
b.backward()
assert abs(a.grad - 6.0) < 1e-9, f"__pow__ 反向梯度应为 6.0，得到 {a.grad}"

# 检查：relu 反向（正数区）
a = Value(2.0)
r = a.relu()
r.backward()
assert a.grad == 1.0, "relu 正数区反向梯度应为 1.0"

# 检查：relu 反向（负数区）
a = Value(-1.0)
r = a.relu()
r.backward()
assert a.grad == 0.0, "relu 负数区反向梯度应为 0.0"

# 检查：tanh 反向
a = Value(0.0)
t = a.tanh()       # tanh(0)=0, 梯度=1-0^2=1
t.backward()
assert abs(a.grad - 1.0) < 1e-9, "tanh(0) 反向梯度应为 1.0"

print("✅ 所有算子前向 & 反向检查通过")

## 参数实验：二维感知机前向传播

用 `Value` 算子搭建一个最小感知机：

```
z = w1*x1 + w2*x2 + b
output = tanh(z)         # 激活函数
```

参数：`w1=0.5, w2=-0.3, b=0.1, x1=1.0, x2=2.0`

**预期现象**：
- `z.data` 应为 `0.5*1 + (-0.3)*2 + 0.1 = 0.0`
- `output.data` 应为 `tanh(0.0) = 0.0`
- 调用 `output.backward()` 后，`w1.grad` 和 `w2.grad` 分别反映各自输入对输出的敏感度
- 把激活换成 `relu` 观察输出差异（`relu(0.0) = 0.0`，但梯度路径不同）

In [ ]:
# 二维感知机前向 + 反向
w1 = Value(0.5)
w2 = Value(-0.3)
b  = Value(0.1)
x1 = Value(1.0)
x2 = Value(2.0)

z      = w1*x1 + w2*x2 + b   # 线性组合
output = z.tanh()              # tanh 激活

print(f"z.data     = {z.data:.4f}      (期望 0.0)")
print(f"output.data= {output.data:.4f}  (期望 tanh(0.0) = 0.0)")

output.backward()
print(f"\n反向传播后：")
print(f"w1.grad = {w1.grad:.4f}   (= x1 * tanh'(z) = 1.0 * 1.0)")
print(f"w2.grad = {w2.grad:.4f}   (= x2 * tanh'(z) = 2.0 * 1.0)")
print(f"b.grad  = {b.grad:.4f}   (= tanh'(z) = 1.0)")

# 对比 relu 激活
w1r, w2r, br = Value(0.5), Value(-0.3), Value(0.1)
zr = w1r*x1 + w2r*x2 + br
out_r = zr.relu()
out_r.backward()
print(f"\nrelu 激活：output={out_r.data:.4f}, w1.grad={w1r.grad:.4f}, w2.grad={w2r.grad:.4f}")
print("（relu 在 z=0 处次梯度取 0，梯度传播截断）")

## 本课收束

本节实现了 `__pow__`、`relu`、`tanh`、`exp` 四个算子，每个算子在返回新 `Value` 节点的同时封装了反向梯度闭包。这些算子的输出 `out.data` 就是下一节将要累积梯度的"局部变量"，梯度通过 `out.grad` 向上游流动。完整的算子集让 `Value` 能描述任意深度的计算图——对应 Aurora 未来 `src/aurora/nn/` 子包中每一层的前向计算。下一节 **M2-A3** 将在这套算子之上实现 `.backward()` 拓扑排序，完成真正的自动微分引擎。